In [ ]:
# 中文:环境自检 —— 打印本 notebook 依赖的关键第三方库版本(huggingface_hub 用于下载权重/分词器文件,
# tokenizers 用于加载 BPE 分词器,torch 用于搭建和运行模型),确保后续步骤的运行环境符合预期
from importlib.metadata import version

pkgs = [
    "huggingface_hub",  # to download pretrained weights
    "tokenizers",       # to implement the tokenizer
    "torch",            # to implement the model
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

1. Architecture code

In [ ]:
# 中文:本单元格定义 Qwen3.5 的完整网络结构,包括:
# 1) FeedForward —— SwiGLU 风格前馈网络
# 2) RMSNorm —— 均方根归一化(Qwen3.5 用 (1+weight) 缩放、weight 零初始化)
# 3) compute_rope_params / apply_rope —— 旋转位置编码(RoPE),支持 partial_rotary_factor(部分旋转)
# 4) GroupedQueryAttention —— 标准的全量(full)自注意力,GQA + QK-Norm + 门控 Q 投影
# 5) TransformerBlock —— 按 layer_type 在 full_attention(上面的 GQA)和
#    linear_attention(Gated DeltaNet,从 qwen3_5_transformers.py 导入)之间切换的 Transformer 块
# 6) Qwen3_5Model —— 整体前向:词嵌入 -> N 层 TransformerBlock -> 最终归一化 -> 输出 logits
import torch
import torch.nn as nn


# 中文:SwiGLU 风格前馈网络。fc1、fc2 都是 emb_dim -> hidden_dim 的投影(分别对应
# "门控分支"和"数值分支"),对 fc1 的输出做 SiLU 激活后与 fc2 的输出逐元素相乘(门控),
# 再用 fc3 投影回 emb_dim。输入/输出张量形状均为 [batch, seq_len, emb_dim]。
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    # 中文:x: [batch, seq_len, emb_dim]
    def forward(self, x):
        # x_fc1, x_fc2: [batch, seq_len, hidden_dim]
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        # 门控:SiLU(x_fc1) * x_fc2,形状仍是 [batch, seq_len, hidden_dim]
        x = nn.functional.silu(x_fc1) * x_fc2
        # 投影回 [batch, seq_len, emb_dim]
        return self.fc3(x)
# 中文:RMSNorm(均方根归一化)——只用均方根做缩放,不像 LayerNorm 那样减均值。
# 注意 Qwen3.5 的写法是 x_norm * (1 + weight),而不是常见的 x_norm * weight,
# 并且 weight 用全零初始化,这样训练刚开始时 (1+weight)=1,等价于不做额外缩放,
# 有利于训练初期的数值稳定性。
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        # Qwen3.5 uses (1 + weight) scaling with zero init
        self.weight = nn.Parameter(torch.zeros(emb_dim))

    # 中文:计算 x 在最后一维上的均方值,取 rsqrt(1/sqrt(.)) 作为缩放系数逐元素相乘;
    # x 的形状不变,例如 [batch, seq_len, emb_dim] 或 [batch, heads, seq_len, head_dim]
    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)

    # 中文:先转 float32 计算归一化(避免 bf16/fp16 下平方和累加的精度损失),
    # 乘以 (1+weight) 做可学习缩放后,再转换回输入原本的 dtype
    def forward(self, x):
        x_norm = self._norm(x.float())
        x_norm = x_norm * (1.0 + self.weight.float())
        return x_norm.to(dtype=x.dtype)
# 中文:预计算 RoPE(旋转位置编码)用的 cos/sin 查找表。
# partial_rotary_factor 控制"部分旋转":只对 head_dim 中的一部分维度施加旋转位置编码,
# 其余维度保持不变——Qwen3.5 用 0.25,即只对 25% 的维度做 RoPE。
def compute_rope_params(
    head_dim,
    theta_base=10_000,
    context_length=4096,
    partial_rotary_factor=1.0,
    dtype=torch.float32,
):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # 中文:实际参与旋转的维度数 = head_dim * partial_rotary_factor,并强制为偶数
    # (RoPE 需要把维度两两配对做旋转)
    rotary_dim = int(head_dim * partial_rotary_factor)
    rotary_dim = max(2, rotary_dim - (rotary_dim % 2))

    # 中文:频率表 inv_freq[i] = theta_base^(-2i/rotary_dim),维度越靠后频率越低,
    # 形状 [rotary_dim/2]
    inv_freq = 1.0 / (
        theta_base ** (
            torch.arange(0, rotary_dim, 2, dtype=dtype)[: (rotary_dim // 2)].float() / rotary_dim
        )
    )

    # 中文:位置索引 0..context_length-1,形状 [context_length]
    positions = torch.arange(context_length, dtype=dtype)
    # 中文:外积得到每个位置、每个频率对应的旋转角度,形状 [context_length, rotary_dim/2]
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)
    # 中文:把角度矩阵复制一份并在最后一维拼接,变成 [context_length, rotary_dim],
    # 这样前一半和后一半维度使用相同的角度,配合 apply_rope 里的 (-x2, x1) 实现标准 RoPE
    angles = torch.cat([angles, angles], dim=1)

    # 中文:预先算好 cos、sin 查找表,形状均为 [context_length, rotary_dim],
    # 推理时按需要的 seq_len 截取,避免重复计算三角函数
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


# 中文:对 query/key 张量应用旋转位置编码。x 形状为 [batch, num_heads, seq_len, head_dim]
def apply_rope(x, cos, sin):
    _, _, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"

    # 中文:rot_dim 即 partial_rotary_factor 对应的实际旋转维度(cos 表的最后一维)
    rot_dim = cos.shape[-1]
    if rot_dim > head_dim:
        raise ValueError(f"RoPE dim {rot_dim} cannot exceed head_dim {head_dim}.")

    # 中文:把 head_dim 切成"参与旋转"(x_rot)和"直接透传、不加位置信息"(x_pass)两部分
    x_rot = x[..., :rot_dim]
    x_pass = x[..., rot_dim:]

    # 中文:x_rot 再对半切成 x1(前半)、x2(后半),用于构造 rotate_half(x)=(-x2, x1)
    x1 = x_rot[..., : rot_dim // 2]
    x2 = x_rot[..., rot_dim // 2 :]

    # 中文:按当前 seq_len 截取 cos/sin,并 unsqueeze 出 batch、head 维,形状变为
    # [1, 1, seq_len, rot_dim],便于和 [batch, num_heads, seq_len, rot_dim] 的 x_rot 广播相乘
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # 中文:标准 RoPE 公式:x_rotated = x_rot*cos + rotate_half(x_rot)*sin
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x_rot * cos) + (rotated * sin)

    # 中文:把旋转后的部分和未旋转的直通部分重新拼接成完整 head_dim,再转回输入的 dtype
    x_out = torch.cat([x_rotated, x_pass], dim=-1)
    return x_out.to(dtype=x.dtype)
# 中文:全量(full)自注意力层,对应 layer_types 中的 "full_attention"。
# 特点:1) GQA —— key/value 头数(num_kv_groups)少于 query 头数(num_heads),多个
#      query 头共享同一组 key/value;2) 可选 QK-Norm —— 对每个 head 的 q/k 向量做 RMSNorm;
# 3) 门控 Q 投影 —— W_query 一次性输出 2 倍维度,一半是真正的 query,一半用来在
#    输出侧做 sigmoid 门控,这是 Qwen3.5 相比传统 GQA 新增的设计。
class GroupedQueryAttention(nn.Module):
    def __init__(
        self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False, dtype=None
    ):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            assert d_in % num_heads == 0, "`d_in` must be divisible by `num_heads` if `head_dim` is not set"
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim

        # 中文:W_query 输出 d_out*2 维,前一半是 query,后一半是门控 gate(见 forward)
        # Qwen3.5 full-attention uses a gated Q projection (2x output dim)
        self.W_query = nn.Linear(d_in, self.d_out * 2, bias=False, dtype=dtype)
        # 中文:key/value 只投影到 num_kv_groups*head_dim(远小于 num_heads*head_dim),
        # 这正是 GQA 节省 KV 缓存/计算量的关键
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)

        self.out_proj = nn.Linear(self.d_out, d_in, bias=False, dtype=dtype)

        # 中文:对每个 head 的 head_dim 向量单独做 RMSNorm(而非整个隐藏维度),
        # 是 Qwen3 系列稳定训练的常见技巧
        if qk_norm:
            self.q_norm = RMSNorm(head_dim, eps=1e-6)
            self.k_norm = RMSNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

    # 中文:forward 输入 x: [batch, num_tokens, d_in];mask: [num_tokens, num_tokens] 的
    # 因果上三角掩码;cos/sin: RoPE 预计算表
    def forward(self, x, mask, cos, sin):
        b, num_tokens, _ = x.shape

        # 中文:一次线性层同时算出 query 和 gate,减少一次矩阵乘法
        q_and_gate = self.W_query(x)
        # reshape 成 [b, num_tokens, num_heads, head_dim*2]
        q_and_gate = q_and_gate.view(b, num_tokens, self.num_heads, self.head_dim * 2)
        # 沿最后一维切成两半:queries、gate,形状均为 [b, num_tokens, num_heads, head_dim]
        queries, gate = torch.chunk(q_and_gate, 2, dim=-1)
        # gate 展平回 [b, num_tokens, d_out],留到最后对输出做门控
        gate = gate.reshape(b, num_tokens, self.d_out)

        # keys/values 用更少的 num_kv_groups 个头,形状 [b, num_tokens, num_kv_groups*head_dim]
        keys = self.W_key(x)
        values = self.W_value(x)

        # 转成 [b, num_heads, num_tokens, head_dim],便于做批量矩阵乘法
        queries = queries.transpose(1, 2)
        # keys/values 转成 [b, num_kv_groups, num_tokens, head_dim]
        keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # 中文:QK-Norm,对 head_dim 维做 RMSNorm
        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys = self.k_norm(keys)

        # 中文:对 query/key 注入旋转位置编码(实际只作用于 partial_rotary_factor 对应的前 1/4 维)
        queries = apply_rope(queries, cos, sin)
        keys = apply_rope(keys, cos, sin)

        # 中文:把 KV 头沿 head 维重复 group_size 次,广播到与 query 头数一致,
        # 这样才能逐头计算注意力(GQA 标准实现方式);重复后 keys/values 变为
        # [b, num_heads, num_tokens, head_dim]
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)

        # 中文:缩放点积注意力打分,形状 [b, num_heads, num_tokens, num_tokens]
        attn_scores = queries @ keys.transpose(2, 3)
        # 中文:因果掩码——mask 中为 True 的位置(未来 token)填 -inf,禁止看到未来
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)
        # 中文:softmax 在 float32 精度下计算以保证数值稳定,算完再转回 queries 的 dtype
        attn_weights = torch.softmax(
            attn_scores * (self.head_dim ** -0.5),
            dim=-1,
            dtype=torch.float32,
        ).to(queries.dtype)

        # 中文:注意力加权求和后,转置回 [b, num_tokens, num_heads, head_dim] 再展平成 [b, num_tokens, d_out]
        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)

        # Qwen3.5 full-attention uses a gated Q projection
        # 中文:用 sigmoid(gate) 对上下文输出做逐元素门控(类似 GLU 结构),
        # 这是 Qwen3.5 full-attention 层相比标准 GQA 新增的机制
        context = context * torch.sigmoid(gate)
        # 中文:最终线性投影回 d_in,输出形状 [b, num_tokens, d_in]
        return self.out_proj(context)
# 中文:从同目录下的 qwen3_5_transformers.py 导入 Qwen3_5GatedDeltaNet —— 这是 Qwen3.5
# "线性注意力"(linear_attention)层的独立实现,即 Gated DeltaNet:用短因果卷积做局部
# 混合,再用门控增量法则(chunk 并行版/recurrent 递归版)以恒定大小的循环状态取代传统
# 注意力的 O(T^2) KV 缓存,兼具线性复杂度和一定的长程记忆能力。核心公式与张量形状的
# 详细中文注释见 qwen3_5_transformers.py 中该类的实现。
# 旁注:此处导入路径是 `from qwen3_5_transformers import ...`,为当前目录下的本地模块,
# 未发现类似 `from Build_an_LLM_from_Scratch....` 这种需要改写成 `from llms_from_scratch....`
# 的错误导入模式。
from qwen3_5_transformers import (
    Qwen3_5GatedDeltaNet,
)

# 中文:HuggingFace transformers 里的 Qwen3_5GatedDeltaNet 期望接收一个"config"对象
# (通过属性访问,而不是 dict),这里写一个轻量适配器类,把 notebook 里用的 cfg 字典
# 字段转换成该类需要的属性名(hidden_size、linear_num_value_heads 等)。
# Just a mapping for the different naming convention in Hugging Face transformers
class _Qwen3_5ConfigAdapter:
    def __init__(self, cfg):
        self.hidden_size = cfg["emb_dim"]
        self.linear_num_value_heads = cfg["linear_num_value_heads"]
        self.linear_num_key_heads = cfg["linear_num_key_heads"]
        self.linear_key_head_dim = cfg["linear_key_head_dim"]
        self.linear_value_head_dim = cfg["linear_value_head_dim"]
        self.linear_conv_kernel_dim = cfg["linear_conv_kernel_dim"]
        self.hidden_act = "silu"
        self.rms_norm_eps = cfg.get("rms_norm_eps", 1e-6)
        self.dtype = cfg.get("dtype", None)


# 中文:Transformer 块——根据 layer_type 选择 token mixer:
# "full_attention" -> 上面定义的 GroupedQueryAttention(标准/门控 GQA + RoPE);
# "linear_attention" -> Qwen3_5GatedDeltaNet(线性复杂度的门控 DeltaNet)。
# 两种层都复用同一套 Pre-Norm 残差结构 + SwiGLU FeedForward。
class TransformerBlock(nn.Module):
    def __init__(self, cfg, layer_type, layer_idx):
        super().__init__()
        self.layer_type = layer_type

        if layer_type == "full_attention":
            self.token_mixer = GroupedQueryAttention(
                d_in=cfg["emb_dim"],
                num_heads=cfg["n_heads"],
                head_dim=cfg["head_dim"],
                num_kv_groups=cfg["n_kv_groups"],
                qk_norm=cfg["qk_norm"],
                dtype=cfg["dtype"],
            )
        elif layer_type == "linear_attention":
            # 中文:Gated DeltaNet 需要 layer_idx,用于在启用 KV/状态缓存时定位属于
            # 本层的循环状态;本 notebook 的前向不使用缓存,layer_idx 只是被传入保存
            self.token_mixer = Qwen3_5GatedDeltaNet(_Qwen3_5ConfigAdapter(cfg), layer_idx)
        else:
            raise ValueError(f"Unsupported layer type: {layer_type}")

        self.ff = FeedForward(cfg)
        self.norm1 = RMSNorm(cfg["emb_dim"], eps=cfg.get("rms_norm_eps", 1e-6))
        self.norm2 = RMSNorm(cfg["emb_dim"], eps=cfg.get("rms_norm_eps", 1e-6))

    # 中文:Pre-Norm 残差结构:先归一化再送入 token mixer,再与残差相加;
    # full_attention 分支需要 mask/cos/sin,linear_attention(Gated DeltaNet)分支
    # 只需要 x 本身——它内部通过短卷积+门控状态更新来建模时序,不需要显式因果 mask 或 RoPE
    def forward(self, x, mask, cos, sin):
        shortcut = x
        x = self.norm1(x)

        # 中文:按层类型分发给对应的 token mixer
        if self.layer_type == "full_attention":
            x = self.token_mixer(x, mask, cos, sin)
        else:
            x = self.token_mixer(x)

        x = x + shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = x + shortcut

        return x
# 中文:完整的 Qwen3.5 语言模型:
# 词嵌入(tok_emb) -> N 层 TransformerBlock(full_attention 与 linear_attention 交替)
# -> 最终 RMSNorm(final_norm) -> 输出线性层(out_head,得到词表 logits)
class Qwen3_5Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

        # 中文:若未显式提供 layer_types,则默认全部使用 full_attention(向后兼容旧配置);
        # Qwen3.5 实际配置里大部分层是 linear_attention(Gated DeltaNet),
        # 每 4 层中固定第 4 层是 full_attention,周期性引入全局注意力
        layer_types = cfg.get("layer_types", ["full_attention"] * cfg["n_layers"])
        if len(layer_types) != cfg["n_layers"]:
            raise ValueError("len(layer_types) must equal n_layers")

        self.trf_blocks = nn.ModuleList(
            [TransformerBlock(cfg, layer_type, idx) for idx, layer_type in enumerate(layer_types)]
        )

        self.final_norm = RMSNorm(cfg["emb_dim"], eps=cfg.get("rms_norm_eps", 1e-6))
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        # 中文:计算 RoPE 需要的 head_dim(仅供 full_attention 层和下面预计算 cos/sin 使用)
        head_dim = cfg["emb_dim"] // cfg["n_heads"] if cfg["head_dim"] is None else cfg["head_dim"]
        # 中文:预先计算好一份全局共享的 RoPE cos/sin 表,按 context_length 算满,
        # forward 时再按实际 seq_len 截取
        cos, sin = compute_rope_params(
            head_dim=head_dim,
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"],
            partial_rotary_factor=cfg.get("partial_rotary_factor", 1.0),
            dtype=torch.float32,
        )
        # 中文:用非 persistent 的 buffer 保存 cos/sin —— 不会写入 state_dict,
        # 但会随 model.to(device) 自动搬到对应设备
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.cfg = cfg

    # 中文:forward 输入 in_idx: [batch, seq_len] 的 token id;返回 logits: [batch, seq_len, vocab_size]
    def forward(self, in_idx):
        x = self.tok_emb(in_idx)

        num_tokens = x.shape[1]
        # 中文:构造 [num_tokens, num_tokens] 的因果上三角掩码(True=需屏蔽的未来位置),
        # 仅供 full_attention 层使用
        mask = torch.triu(
            torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool),
            diagonal=1,
        )

        # 中文:依次通过每一层;linear_attention 层会忽略传入的 mask/cos/sin 参数
        for block in self.trf_blocks:
            x = block(x, mask, self.cos, self.sin)

        # 中文:最终归一化
        x = self.final_norm(x)
        # 中文:转换到目标 dtype(如 bf16)后再做输出投影,与 out_head 权重 dtype 对齐;
        # logits 形状为 [batch, seq_len, vocab_size]
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits

2. Initialize model

In [ ]:
# 中文:Qwen3.5-0.8B 文本模型的配置字典,字段基本对应 HuggingFace 官方 config.json
# (命名做了适配,方便和上面手写的 Qwen3_5Model 对应)。
# Qwen3.5-0.8B text configuration
QWEN3_5_CONFIG = {
    "vocab_size": 248_320,
    "context_length": 262_144,
    "emb_dim": 1_024,
    # n_heads: full_attention 层的 query 头数
    "n_heads": 8,
    "n_layers": 24,
    # hidden_dim: FeedForward(SwiGLU)中间层维度
    "hidden_dim": 3_584,
    # head_dim: 每个注意力头的维度;注意 n_heads*head_dim(8*256=2048)大于 emb_dim(1024),
    # 这是 Qwen3 系列常见设计——注意力内部投影维度可以独立于 emb_dim 选择
    "head_dim": 256,
    "qk_norm": True,
    # n_kv_groups: GQA 的 KV 头组数;group_size = n_heads // n_kv_groups = 4,
    # 即每 4 个 query 头共享 1 组 key/value
    "n_kv_groups": 2,
    # rope_base: RoPE 频率基数 theta,取值很大(1e7)以适配 262144 token 的超长上下文
    "rope_base": 10_000_000.0,
    # partial_rotary_factor: 只对 head_dim 的 25%(64 维)做旋转位置编码,
    # 其余 75% 维度不附加位置信息
    "partial_rotary_factor": 0.25,
    "rms_norm_eps": 1e-6,
    # 中文:以下是线性注意力层(Gated DeltaNet)专用的超参数
    "linear_conv_kernel_dim": 4,
    # linear_key_head_dim: Gated DeltaNet 中 key/query 每个头的维度
    "linear_key_head_dim": 128,
    # linear_value_head_dim: Gated DeltaNet 中 value 每个头的维度
    "linear_value_head_dim": 128,
    # linear_num_key_heads: Gated DeltaNet 的 key/query 头数
    "linear_num_key_heads": 16,
    # linear_num_value_heads: Gated DeltaNet 的 value 头数;与 key 头数相同(均为16),
    # 因此该层内部不需要像 full_attention 那样对 key/value 做重复(repeat_interleave)
    "linear_num_value_heads": 16,
    "dtype": torch.bfloat16,
    # layer_types: 显式指定每一层类型——linear_attention(Gated DeltaNet,线性复杂度)
    # 与 full_attention(标准/门控 GQA)交替,每 4 层里第 4 层是 full_attention
    "layer_types": [
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
        "linear_attention", "linear_attention", "linear_attention", "full_attention",
    ],
}
# 中文:固定随机种子,使随机初始化(如 Gated DeltaNet 中 A_log 的采样)可复现
torch.manual_seed(123)
# 中文:实例化模型——此时权重仍是随机初始化,尚未加载 HuggingFace 预训练权重
model = Qwen3_5Model(QWEN3_5_CONFIG)

In [ ]:
# 中文:直接对 model 求值,让 Jupyter 打印出 nn.Module 的结构(__repr__),
# 用于人工检查各层类型/参数形状是否符合预期
model

In [ ]:
# 中文:用一个长度为 3 的示例序列 [1, 2, 3](batch=1)跑一次前向传播,
# 仅用来检验 forward 能否跑通,并不使用返回的 logits
model(torch.tensor([1, 2, 3]).unsqueeze(0))

# 中文:统计模型全部参数量;若 tok_emb 与 out_head 权重被绑定(tie)为同一份张量,
# 这里会被重复计入两次
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# Account for weight tying
# 若采用权重绑定(weight tying),减去一次 embedding 参数量,得到"唯一"参数量
total_params_normalized = total_params - model.tok_emb.weight.numel()
print(f"\nTotal number of unique parameters: {total_params_normalized:,}")

In [ ]:
# 中文:估算模型在给定 dtype 下,参数 + 梯度 + buffer 理论上占用的显存/内存大小(GB)。
# 注意这只是粗略估算,不包含推理/训练时的激活值、优化器状态等额外开销。
def calc_model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # Calculate total number of elements per parameter
        param_size = param.numel()
        total_params += param_size
        # Check if gradients are stored for this parameter
        if param.requires_grad:
            total_grads += param_size

    # Calculate buffer size (non-parameters that require memory)
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # Size in bytes = (Number of elements) * (Size of each element in bytes)
    # We assume parameters and gradients are stored in the same type as input dtype
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # Convert bytes to gigabytes
    total_memory_gb = total_memory_bytes / (1024**3)

    return total_memory_gb

print(f"float32 (PyTorch default): {calc_model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {calc_model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

In [ ]:
# 中文:按优先级选择可用计算设备:CUDA(NVIDIA GPU) > MPS(Apple Silicon) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# 中文:把模型搬到选定设备;末尾分号用于抑制 Jupyter 对表达式返回值的自动打印
model.to(device);

3. Load pretrained weights

In [ ]:
# 中文:把 HuggingFace safetensors 检查点里、按 HF 命名的原始权重,逐一拷贝进
# 当前这个用纯 PyTorch 手写的 Qwen3_5Model(按本 notebook 的命名)对应参数中。
def load_weights_into_qwen3_5(model, param_config, params):
    # 中文:内部工具函数——校验形状一致后,把右侧权重原地拷贝进左侧参数张量,
    # 返回该参数(仍是同一个 nn.Parameter 对象,不改变模型的 state_dict 结构)
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(
                f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}"
            )

        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right)
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

        return left

    # 中文:不同版本/不同打包方式的检查点,权重 key 的前缀可能是 "model."
    # 或 "model.language_model."(例如多模态模型里语言模型是子模块),这里做兼容判断
    if "model.embed_tokens.weight" in params:
        model_prefix = "model"
    elif "model.language_model.embed_tokens.weight" in params:
        model_prefix = "model.language_model"
    else:
        raise KeyError("Could not find embed token weights in checkpoint.")

    def pkey(suffix):
        return f"{model_prefix}.{suffix}"

    # 中文:加载词嵌入表,形状 [vocab_size, emb_dim]
    model.tok_emb.weight = assign(
        model.tok_emb.weight,
        params[pkey("embed_tokens.weight")],
        pkey("embed_tokens.weight"),
    )

    n_layers = param_config["n_layers"]
    layer_types = param_config.get("layer_types", ["full_attention"] * n_layers)

    # 中文:逐层拷贝权重;根据 layer_types[l] 判断这一层是 full_attention 还是
    # linear_attention,两者的权重命名和结构完全不同
    for l in range(n_layers):
        block = model.trf_blocks[l]
        layer_type = layer_types[l]

        # 中文:full_attention 层——拷贝 GroupedQueryAttention 的
        # q_proj/k_proj/v_proj/o_proj,以及可选的 q_norm/k_norm(QK-Norm)
        if layer_type == "full_attention":
            att = block.token_mixer
            att.W_query.weight = assign(
                att.W_query.weight,
                params[pkey(f"layers.{l}.self_attn.q_proj.weight")],
                pkey(f"layers.{l}.self_attn.q_proj.weight"),
            )
            att.W_key.weight = assign(
                att.W_key.weight,
                params[pkey(f"layers.{l}.self_attn.k_proj.weight")],
                pkey(f"layers.{l}.self_attn.k_proj.weight"),
            )
            att.W_value.weight = assign(
                att.W_value.weight,
                params[pkey(f"layers.{l}.self_attn.v_proj.weight")],
                pkey(f"layers.{l}.self_attn.v_proj.weight"),
            )
            att.out_proj.weight = assign(
                att.out_proj.weight,
                params[pkey(f"layers.{l}.self_attn.o_proj.weight")],
                pkey(f"layers.{l}.self_attn.o_proj.weight"),
            )
            if hasattr(att, "q_norm") and att.q_norm is not None:
                att.q_norm.weight = assign(
                    att.q_norm.weight,
                    params[pkey(f"layers.{l}.self_attn.q_norm.weight")],
                    pkey(f"layers.{l}.self_attn.q_norm.weight"),
                )
            if hasattr(att, "k_norm") and att.k_norm is not None:
                att.k_norm.weight = assign(
                    att.k_norm.weight,
                    params[pkey(f"layers.{l}.self_attn.k_norm.weight")],
                    pkey(f"layers.{l}.self_attn.k_norm.weight"),
                )

        # 中文:linear_attention 层(Gated DeltaNet)——拷贝其专属参数:
        # dt_bias/A_log(衰减相关标量)、conv1d(短因果卷积)、norm(门控 RMSNorm)、
        # out_proj,以及 4 个输入投影 in_proj_qkv/in_proj_z/in_proj_b/in_proj_a
        elif layer_type == "linear_attention":
            lat = block.token_mixer
            # dt_bias: 每个 value head 一个标量,用于计算衰减系数
            # g = -exp(A_log) * softplus(a + dt_bias)
            lat.dt_bias = assign(
                lat.dt_bias,
                params[pkey(f"layers.{l}.linear_attn.dt_bias")],
                pkey(f"layers.{l}.linear_attn.dt_bias"),
            )
            # A_log = log(A),同样每个 value head 一个标量,共同决定该层的遗忘/衰减速度
            lat.A_log = assign(
                lat.A_log,
                params[pkey(f"layers.{l}.linear_attn.A_log")],
                pkey(f"layers.{l}.linear_attn.A_log"),
            )
            # 深度可分离短因果卷积权重,形状 [conv_dim, 1, kernel_size],
            # 对 QKV 混合向量做局部时序平滑
            lat.conv1d.weight = assign(
                lat.conv1d.weight,
                params[pkey(f"layers.{l}.linear_attn.conv1d.weight")],
                pkey(f"layers.{l}.linear_attn.conv1d.weight"),
            )
            # Gated RMSNorm 权重,用于 Gated DeltaNet 输出前的门控归一化
            lat.norm.weight = assign(
                lat.norm.weight,
                params[pkey(f"layers.{l}.linear_attn.norm.weight")],
                pkey(f"layers.{l}.linear_attn.norm.weight"),
            )
            # 线性注意力层输出投影,把 value_dim 映射回 hidden_size(emb_dim)
            lat.out_proj.weight = assign(
                lat.out_proj.weight,
                params[pkey(f"layers.{l}.linear_attn.out_proj.weight")],
                pkey(f"layers.{l}.linear_attn.out_proj.weight"),
            )
            # 一次性投影出拼接的 query/key/value 向量(经短因果卷积后再拆分)
            lat.in_proj_qkv.weight = assign(
                lat.in_proj_qkv.weight,
                params[pkey(f"layers.{l}.linear_attn.in_proj_qkv.weight")],
                pkey(f"layers.{l}.linear_attn.in_proj_qkv.weight"),
            )
            # 门控分支 z 的投影,供 Gated RMSNorm 做逐元素缩放
            lat.in_proj_z.weight = assign(
                lat.in_proj_z.weight,
                params[pkey(f"layers.{l}.linear_attn.in_proj_z.weight")],
                pkey(f"layers.{l}.linear_attn.in_proj_z.weight"),
            )
            # beta(写入强度)输入 b 的投影
            lat.in_proj_b.weight = assign(
                lat.in_proj_b.weight,
                params[pkey(f"layers.{l}.linear_attn.in_proj_b.weight")],
                pkey(f"layers.{l}.linear_attn.in_proj_b.weight"),
            )
            # 衰减系数相关输入 a 的投影
            lat.in_proj_a.weight = assign(
                lat.in_proj_a.weight,
                params[pkey(f"layers.{l}.linear_attn.in_proj_a.weight")],
                pkey(f"layers.{l}.linear_attn.in_proj_a.weight"),
            )

        else:
            raise ValueError(f"Unsupported layer type: {layer_type}")

        # 中文:以下是两种层类型共用的部分——Pre-Norm 的两个 RMSNorm
        # 以及 SwiGLU FeedForward 的三个权重
        block.norm1.weight = assign(
            block.norm1.weight,
            params[pkey(f"layers.{l}.input_layernorm.weight")],
            pkey(f"layers.{l}.input_layernorm.weight"),
        )

        block.ff.fc1.weight = assign(
            block.ff.fc1.weight,
            params[pkey(f"layers.{l}.mlp.gate_proj.weight")],
            pkey(f"layers.{l}.mlp.gate_proj.weight"),
        )
        block.ff.fc2.weight = assign(
            block.ff.fc2.weight,
            params[pkey(f"layers.{l}.mlp.up_proj.weight")],
            pkey(f"layers.{l}.mlp.up_proj.weight"),
        )
        block.ff.fc3.weight = assign(
            block.ff.fc3.weight,
            params[pkey(f"layers.{l}.mlp.down_proj.weight")],
            pkey(f"layers.{l}.mlp.down_proj.weight"),
        )
        block.norm2.weight = assign(
            block.norm2.weight,
            params[pkey(f"layers.{l}.post_attention_layernorm.weight")],
            pkey(f"layers.{l}.post_attention_layernorm.weight"),
        )

    # 中文:加载最终归一化层权重
    model.final_norm.weight = assign(
        model.final_norm.weight,
        params[pkey("norm.weight")],
        pkey("norm.weight"),
    )

    # 中文:输出层权重——优先用独立的 lm_head;若检查点没有单独的 lm_head,
    # 说明该模型采用权重绑定(tie word embeddings),直接复用 tok_emb 的权重
    if "lm_head.weight" in params:
        model.out_head.weight = assign(model.out_head.weight, params["lm_head.weight"], "lm_head.weight")
    elif pkey("lm_head.weight") in params:
        model.out_head.weight = assign(model.out_head.weight, params[pkey("lm_head.weight")], pkey("lm_head.weight"))
    else:
        model.out_head.weight = model.tok_emb.weight
        print("Model uses weight tying.")
# 中文:以下开始真正下载并加载 Qwen3.5-0.8B 的预训练权重
import json
import os
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, snapshot_download

# HuggingFace Hub 上的仓库 ID
repo_id = "Qwen/Qwen3.5-0.8B"
local_dir = Path(repo_id).parts[-1]

# 下载整个仓库快照到本地目录(包含 safetensors 分片文件和索引文件)
repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)
# 读取 safetensors 的分片索引,得知每个权重张量具体存放在哪个分片文件里
index_path = os.path.join(repo_dir, "model.safetensors.index.json")
with open(index_path, "r") as f:
    index = json.load(f)

# 依次加载所有分片文件,合并成一个完整的 {权重名: 张量} 字典
weights_dict = {}
for filename in sorted(set(index["weight_map"].values())):
    shard_path = os.path.join(repo_dir, filename)
    shard = load_file(shard_path)
    weights_dict.update(shard)

# 把下载好的权重拷贝进我们自己实现的模型
load_weights_into_qwen3_5(model, QWEN3_5_CONFIG, weights_dict)
# 加载完成后再次确保模型在目标设备上(safetensors 读出来的张量默认在 CPU)
model.to(device)
# 权重已拷贝进模型参数,释放这份临时字典以节省内存
del weights_dict

4. Load tokenizer

In [ ]:
# 中文:定义一个轻量级的 Qwen3.5 分词器封装类。底层用 HuggingFace tokenizers 库的
# BPE 分词器处理普通文本,同时手工识别一批不能被 BPE 拆开的"特殊 token"
# (聊天/多模态控制符),并实现 Qwen3.5 的 chat template 包装逻辑。
import re
from tokenizers import Tokenizer


# 中文:需要整体识别、不能被普通 BPE 切分的特殊 token 列表
class Qwen3_5Tokenizer:
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>", "<|im_end|>",
        "<|object_ref_start|>", "<|object_ref_end|>",
        "<|box_start|>", "<|box_end|>",
        "<|quad_start|>", "<|quad_end|>",
        "<|vision_start|>", "<|vision_end|>",
        "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
        "<think>", "</think>",
    ]
    # 中文:先用正则把文本切成"特殊 token 片段"和"普通文本片段";
    # 普通片段交给底层 BPE 分词器编码,特殊 token 片段直接查表得到对应 id
    _SPLIT_RE = re.compile(r"(<\|[^>]+?\|>|<think>|</think>)")

    # 中文:加载 tokenizer.json(词表 + BPE 合并规则),并建立
    # "特殊 token 字符串 -> id" 的映射表
    def __init__(
        self,
        tokenizer_file_path="tokenizer.json",
        repo_id=None,
        apply_chat_template=True,
        add_generation_prompt=False,
        add_thinking=False,
    ):
        self.apply_chat_template = apply_chat_template
        self.add_generation_prompt = add_generation_prompt
        self.add_thinking = add_thinking

        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        self._special_to_id = {}
        for t in self._SPECIALS:
            tid = self._tok.token_to_id(t)
            if tid is not None:
                self._special_to_id[t] = tid

        # 用 <|endoftext|> 兼作 pad token 和默认的 eos token
        self.pad_token_id = self._special_to_id["<|endoftext|>"]
        self.eos_token_id = self.pad_token_id

        # 非 Base(即 Chat/Instruct)模型用 <|im_end|> 作为真正的结束符;
        # Base 模型仍用 <|endoftext|>
        if repo_id and "Base" not in repo_id:
            eos_token = "<|im_end|>"
        else:
            eos_token = "<|endoftext|>"
        if eos_token in self._special_to_id:
            self.eos_token_id = self._special_to_id[eos_token]

    # 中文:编码——可选先按 chat template 包装 prompt,再用正则切分
    # 特殊 token 与普通文本分别处理
    def encode(self, text, chat_wrapped=None):
        if chat_wrapped is None:
            chat_wrapped = self.apply_chat_template

        # 特殊情况:如果整段输入本身就是一个特殊 token(且不含换行),
        # 直接返回其 id,不做 chat 包装/分词
        stripped = text.strip()
        if stripped in self._special_to_id and "\n" not in stripped:
            return [self._special_to_id[stripped]]

        if chat_wrapped:
            text = self._wrap_chat(text)

        # 依次遍历切分出的片段:是特殊 token 就查表取 id,否则交给底层 BPE 分词器编码
        ids = []
        for part in filter(None, self._SPLIT_RE.split(text)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    # 中文:解码时保留特殊 token(skip_special_tokens=False),
    # 便于观察 chat 模板/思考标签等结构是否符合预期
    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)

    # 中文:按 Qwen3.5 的 chat template 手工拼接 user 消息 + assistant 生成前缀
    # (可选加入 <think> 思考标签起始,让模型进入"思考模式"生成)
    def _wrap_chat(self, user_msg):
        # Mirrors Qwen3.5 chat_template behavior:
        # add_generation_prompt + thinking => "<think>\n"
        # add_generation_prompt + no thinking => empty think scaffold
        s = f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        if self.add_generation_prompt:
            s += "<|im_start|>assistant\n"
            if self.add_thinking:
                s += "<think>\n"
            else:
                s += "<think>\n\n</think>\n\n"
        return s
# 中文:下载并加载真实的分词器文件(词表 + 合并规则)
tokenizer_file_path = "Qwen3.5-0.8B/tokenizer.json"

# 下载 tokenizer.json 到本地目录
hf_hub_download(
    repo_id=repo_id,
    filename="tokenizer.json",
    local_dir=local_dir,
)

# 中文:add_generation_prompt=True 会在末尾拼接 "<|im_start|>assistant\n";
# add_thinking=True 会进一步加上 "<think>\n",让模型以思考模式开始生成
tokenizer = Qwen3_5Tokenizer(
    tokenizer_file_path=tokenizer_file_path,
    repo_id=repo_id,
    apply_chat_template=True,
    add_generation_prompt=True,
    add_thinking=True,
)
# 用一个示例问题测试编码/解码是否正确往返
# (先编码再解码,应能看到完整的 chat 模板文本,包括 <|im_start|>/<think> 等标签)
prompt = "Give me a short introduction to large language models."

input_token_ids = tokenizer.encode(prompt)
text = tokenizer.decode(input_token_ids)
text

4. Generate text

In [ ]:
# 中文:最基础的贪心解码生成函数——逐 token yield 结果,不使用 KV 缓存,
# 因此每一步都要把当前已生成的完整序列重新喂给模型做一次前向传播
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):

    # 切换到 eval 模式(本模型未用到 dropout,但仍是好习惯)
    model.eval()
    with torch.no_grad():
        # 最多生成 max_new_tokens 个 token
        for _ in range(max_new_tokens):
            # 中文:没有 KV 缓存,所以每步都要把全部 token_ids 重新前向一遍;
            # 只取序列最后一个位置的 logits,形状 [batch, vocab_size]
            out = model(token_ids)[:, -1]
            # 贪心解码——直接取概率最大的 token id,不做随机采样
            next_token = torch.argmax(out, dim=-1, keepdim=True)

            # 若生成到结束符(eos),提前终止生成
            if (eos_token_id is not None
                   and torch.all(next_token == eos_token_id)):
               break

            # 用生成器逐 token 吐出结果,方便流式打印
            yield next_token

            # 把新生成的 token 拼接到序列末尾,作为下一步的输入
            token_ids = torch.cat([token_ids, next_token], dim=1)
# 中文:生成示例一——让模型续写一段介绍大语言模型的文本
import time

prompt = "Give me a short introduction to large language models."

# 编码 prompt(内部会按 chat template 包装并加上 <think> 前缀)
input_token_ids = tokenizer.encode(prompt)
# 转成张量并添加 batch 维,形状为 [1, prompt_len]
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)

# 若在 CUDA 上运行,重置显存峰值统计,方便之后测量本次生成实际占用的显存
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()
generated_tokens = 0

# 逐 token 生成并实时打印,同时统计生成速度(tokens/sec)
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=500,
    eos_token_id=tokenizer.eos_token_id
):
    generated_tokens += 1
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

elapsed = time.perf_counter() - start_time
tokens_per_sec = generated_tokens / elapsed if elapsed > 0 else 0.0
print(f"\n\nGeneration speed: {tokens_per_sec:.2f} tokens/sec")

if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"GPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")

In [ ]:
# 中文:生成示例二——给模型一道简单的数学应用题(先打八折/九折等折扣、再加税),
# 用于直观感受模型的推理/思维链能力;流程与上一个单元格完全一致
import time

prompt = "A shop gives a 20% discount, then adds 10% tax. Is the final price higher or lower than the original? By how much?"

# 编码 prompt(同样会按 chat template 包装并加上 <think> 前缀)
input_token_ids = tokenizer.encode(prompt)
# 转成张量并添加 batch 维,形状为 [1, prompt_len]
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)

# 若在 CUDA 上运行,重置显存峰值统计
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()
generated_tokens = 0

# 逐 token 生成并实时打印,同时统计生成速度(tokens/sec)
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=500,
    eos_token_id=tokenizer.eos_token_id
):
    generated_tokens += 1
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

elapsed = time.perf_counter() - start_time
tokens_per_sec = generated_tokens / elapsed if elapsed > 0 else 0.0
print(f"\n\nGeneration speed: {tokens_per_sec:.2f} tokens/sec")

if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"GPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")